# ⏱️ 08 - Timing and Performance

In data science, performance matters. Code that's instant on 100 rows can take minutes on 10 million. Jupyter/IPython give you tools to **measure** how long code takes, find **where** the time goes, and check how much **memory** your data uses.

## ✅ Learning Goals
- Use `%time` and `%%time` for a quick, one-time measurement
- Use `%timeit` and `%%timeit` for reliable comparisons, and read their output
- Compare two approaches **fairly**: same input, same result, setup excluded
- Compare Python loops with NumPy **vectorized** operations
- Spot repeated work, and see how runtime grows with data size
- Find the slow part of a function with `%prun` (and `%lprun`)
- Measure memory with `nbytes` and `memory_usage`, and load data efficiently with Pandas

> **The core habit:** measure first, improve the bottleneck, then measure again.

## Setup
All imports go in the first cell (notebook 02). This notebook creates one data file, in `08-OutputFiles`.

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

OUT = Path('08-OutputFiles')
OUT.mkdir(exist_ok=True)

## 1. Why Measure?
Performance work should start with **evidence**, not a hunch:

```text
Feels slow → measure → find the bottleneck → improve it → measure again
```

Don't optimize code just because it **looks** slow. A loop may be fine for 100 values and painful for 10 million, and a clever-looking rewrite may not help the real workload at all.

Useful questions:
- Which cell takes the longest? Which function inside it?
- Does the runtime grow too quickly as the data grows?
- Is the code doing the **same work** over and over?
- Could NumPy or Pandas do the operation **in bulk** instead of one item at a time?

## 2. `%time`: One Quick Measurement
`%time` runs a statement **once** and reports how long it took. It works on a line with an assignment, too.

In [2]:
numbers = list(range(1_000_000))

%time total = sum(numbers)

CPU times: total: 15.6 ms
Wall time: 5.85 ms


The output shows two kinds of time:
- **Wall time** is real "clock on the wall" time, how long **you** waited. This is usually the number you care about.
- **CPU times** are how long the processor was actually busy. Wall time is much bigger than CPU time when the code is **waiting**, for example on a file, a website, or a slow disk.

✅ **Your Turn**: Use `%time` to measure summing one million integers with `np.sum(np.arange(1_000_000))`. Run the cell three times. Do you get the exact same time each run?

In [3]:
# Your turn here
%time np.sum(np.arange(1_000_000))

CPU times: total: 31.2 ms
Wall time: 2.8 ms


np.int64(499999500000)

No, I do not get the exact same time every run.
The execution time changes slightly because of background
processes, computer activity, and other factors.

## 3. `%timeit`: Reliable Measurements
A single timing can be thrown off by other programs, background work, or one-time setup costs. `%timeit` runs an expression **many times** and reports the average. That's much more reliable for comparing two approaches.

In [4]:
numbers = list(range(1_000))
%timeit [x**2 for x in numbers]

37.2 μs ± 241 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


### Reading the Output
```text
41.2 μs ± 1.05 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
```
- **10,000 loops each**: in each run, the code was repeated 10,000 times. `%timeit` picks this count automatically, so fast code gets more loops.
- **7 runs**: that was done 7 separate times.
- **41.2 μs per loop**: on average, **one** execution took 41.2 microseconds.
- **± 1.05 μs**: the standard deviation, which shows how much the runs varied. Small is good.

| Unit | Meaning |
|---|---|
| s | second |
| ms | millisecond = 1/1,000 s |
| μs (or µs) | microsecond = 1/1,000,000 s |
| ns | nanosecond = 1/1,000,000,000 s |

### Saving the Result with `-o`
`-o` returns the timing as an object, so you can do math with it. For example, you can work out **how many times faster** one approach is:

In [5]:
t_comp = %timeit -o [n * 2 for n in range(10_000)]
t_map = %timeit -o list(map(lambda n: n * 2, range(10_000)))

print(f'The comprehension is {t_map.average / t_comp.average:.2f}x as fast as map + lambda')

308 μs ± 1.82 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
676 μs ± 36 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
The comprehension is 2.19x as fast as map + lambda


💡 For slow code, control the repetitions yourself: `%timeit -n 10 -r 3 ...` means 10 loops per run, 3 runs.

✅ **Your Turn**: Use `%timeit -o` to compare `sorted(data)` with `data.sort()` on a fresh copy, using `data = list(range(100_000, 0, -1))`. *Hint:* `data.sort()` changes the list in place, so time `data.copy().sort()` and `sorted(data.copy())` to make the comparison fair.

In [6]:
# Your turn here
data = list(range(100_000, 0, -1))

# Time sorted()
t_sorted = %timeit -o sorted(data.copy())

# Time sort()
t_sort = %timeit -o data.copy().sort()

print(f"Sorted(): {t_sorted.average:.6f} seconds")
print(f"Sort(): {t_sort.average:.6f} seconds")

if t_sorted.average < t_sort.average:
    print("sorted() is faster.")
else:
    print("sort() is faster.")

1.01 ms ± 15.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
530 μs ± 39.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Sorted(): 0.001006 seconds
Sort(): 0.000530 seconds
sort() is faster.


## 4. `%%time` and `%%timeit` for a Whole Cell
Put `%%time` or `%%timeit` as the **first line** of a cell to measure everything in it, not just one expression.

| Magic | Times... | Runs it... |
|---|---|---|
| `%time` | One line | Once |
| `%%time` | The whole cell | Once |
| `%timeit` | One line | Many times |
| `%%timeit` | The whole cell | Many times |

In [7]:
%%timeit
total = 0
for i in range(1_000_000):
    total += i

32.3 ms ± 997 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


⚠️ **Gotcha:** variables created inside a `%%timeit` cell **don't exist afterwards**. The cell ran in a temporary space, many times over. `%%time` runs your code normally, so its variables stay:

In [8]:
%%time
squares = [value ** 2 for value in range(100_000)]

CPU times: total: 0 ns
Wall time: 5.52 ms


In [9]:
len(squares)   # works, because %%time keeps variables. After %%timeit this would be a NameError

100000

## 5. Comparing Fairly
A timing comparison is only meaningful when both versions:
- use the **same input** (and the same size),
- do the **same work**, with no extra setup hiding in one of them,
- produce the **same result** (check it!),
- and are measured over **multiple runs** (`%timeit`).

**Unfair:** suppose your data starts as a list. The first line below also **converts** the list to an array every time, and the second doesn't. The difference is huge, and it has nothing to do with squaring:

In [10]:
data_list = list(range(1_000_000))
data_array = np.array(data_list)

%timeit np.array(data_list) ** 2    # converts the list AND squares it
%timeit data_array ** 2             # only squares it

35.5 ms ± 629 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
2.02 ms ± 179 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


**Fair:** create the input **once**, outside the timing, then time only the operation you care about. And before comparing speed, confirm that both versions give the same answer:

In [11]:
values_list = list(range(1_000_000))
values_array = np.arange(1_000_000)

loop_result = [v ** 2 for v in values_list]
numpy_result = values_array ** 2

np.array_equal(np.array(loop_result), numpy_result)

True

✅ **Your Turn**: Time `sum(range(10_000))` against `np.sum(np.arange(10_000))`. Then time `np.sum(arr)` where `arr = np.arange(10_000)` was created **beforehand**. How much of NumPy's time was spent just creating the array? In a comment, explain which comparison is fair for which question.

In [12]:
# Your turn here
# Compare Python sum()
%timeit sum(range(10_000))

# Compare NumPy including array creation
%timeit np.sum(np.arange(10_000))

# Create the array before measuring
arr = np.arange(10_000)

# Measure only NumPy sum
%timeit np.sum(arr)

118 μs ± 7.27 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
7.21 μs ± 119 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
3.5 μs ± 55.8 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


The first comparison measures Python's sum against NumPy's sum
including the time needed to create the NumPy array.

The second NumPy measurement only measures the summation because
the array was created beforehand.

The comparison between sum(range(10_000)) and np.sum(arr) is
fairer when comparing the summation operations themselves.

The difference between the two NumPy measurements shows the
additional time spent creating the array.

## 6. Python Loops vs. NumPy
This is the payoff. NumPy (notebook 07) does the same calculation **in bulk**:
- **Python loop:** Python handles one value at a time, checking its type and looking up `**` on every pass.
- **NumPy (vectorized):** Python makes **one** call, and NumPy runs the whole loop in fast, compiled code.

### Squaring

In [13]:
numbers_list = list(range(1_000_000))
numbers_array = np.arange(1_000_000)

def square_with_loop(values):
    result = []
    for value in values:
        result.append(value ** 2)
    return result

def square_with_numpy(values):
    return values ** 2

t_loop = %timeit -o square_with_loop(numbers_list)
t_numpy = %timeit -o square_with_numpy(numbers_array)
print(f'NumPy is about {t_loop.average / t_numpy.average:.0f}x faster')

67.6 ms ± 938 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
1.84 ms ± 23.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
NumPy is about 37x faster


### Filtering
A **boolean mask** (`values[values >= 700_000]`, from notebook 07) replaces a loop with an `if` inside:

In [14]:
def passing_loop(values):
    result = []
    for score in values:
        if score >= 700_000:
            result.append(score)
    return result

def passing_numpy(values):
    return values[values >= 700_000]

%timeit passing_loop(numbers_list)
%timeit passing_numpy(numbers_array)

24.5 ms ± 268 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
772 μs ± 40.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### Summary Statistics

In [15]:
def mean_with_loop(values):
    total = 0
    for value in values:
        total += value
    return total / len(values)

%timeit mean_with_loop(numbers_list)
%timeit np.mean(numbers_array)

22.3 ms ± 969 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
617 μs ± 74.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


Speed isn't the only win. `values[values >= 700_000]` is also **shorter and clearer** than six lines of loop. It says *what* you want rather than *how* to do it.

⚠️ Looping **over a NumPy array** with a plain `for` loop is even slower than looping over a list, because each element is converted to a Python object one at a time. Arrays are fast only when NumPy does the looping.

✅ **Your Turn**:
1. Write a loop that adds 10 to every value in `numbers_list`, and a NumPy version for `numbers_array`. Check that the results match, then time both.
2. Do the same for selecting the values **greater than 500,000**.

In [17]:
# Your turn here
# Python loop
loop_result = []

for value in numbers_list:
    loop_result.append(value + 10)

# NumPy version
numpy_result = numbers_array + 10

# Check if both results match
print(np.array_equal(np.array(loop_result), numpy_result))

# Time both approaches
%timeit [value + 10 for value in numbers_list]
%timeit numbers_array + 10

True
51.7 ms ± 10.3 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
1.89 ms ± 78.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [18]:
# Python loop
loop_result = []

for value in numbers_list:
    if value > 500_000:
        loop_result.append(value)

# NumPy version
numpy_result = numbers_array[numbers_array > 500_000]

# Check if both results match
print(np.array_equal(np.array(loop_result), numpy_result))

# Time both approaches
%timeit [value for value in numbers_list if value > 500_000]
%timeit numbers_array[numbers_array > 500_000]

True
25.5 ms ± 189 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
1.08 ms ± 89.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## 7. Repeated Work and How Runtime Grows
A very common slowdown is recalculating the **same thing inside a loop**. Here, every score is divided by the top score, but `max(scores)` is recomputed on **every** pass:

In [19]:
def normalize_slow(scores):
    return [s / max(scores) for s in scores]   # max() scans the whole list, every time

def normalize_fast(scores):
    top = max(scores)                          # compute it once
    return [s / top for s in scores]

scores = list(np.random.randint(0, 100, size=2_000))
print(normalize_slow(scores) == normalize_fast(scores))   # same result?

%timeit normalize_slow(scores)
%timeit normalize_fast(scores)

True
48.7 ms ± 340 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
122 μs ± 653 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


### Watch How It Scales
Now **double** the data a few times. `normalize_fast` takes about **2x** as long each time (it's *linear*). `normalize_slow` takes about **4x** as long, because it does *n* passes of *n* steps each (it's *quadratic*). On large data, that difference is everything:

In [20]:
for n in [1_000, 2_000, 4_000]:
    data = list(range(1, n + 1))
    t_slow = %timeit -o -q -n 3 -r 3 normalize_slow(data)
    t_fast = %timeit -o -q -n 3 -r 3 normalize_fast(data)
    print(f'n={n:>5}:  slow {t_slow.average * 1000:8.2f} ms   fast {t_fast.average * 1000:6.3f} ms')

n= 1000:  slow     8.67 ms   fast  0.045 ms
n= 2000:  slow    31.67 ms   fast  0.088 ms
n= 4000:  slow   128.31 ms   fast  0.173 ms


(`-q` keeps `%timeit` quiet, so only the `print` shows.)

Ask yourself: *Am I recalculating the same max, filter, conversion, or **file read** inside a loop?* If so, do it once, before the loop.

✅ **Your Turn**: Write a loop that recomputes something it doesn't need to, such as `len(my_list)` or `sum(my_list)` inside every pass. Move it outside the loop, and use `%timeit` to compare the two versions.

In [21]:
# Your turn here

my_list = list(range(10_000))

# Slow version: calculates the sum every time
def slow_sum():
    total = 0

    for i in range(len(my_list)):
        total += sum(my_list)

    return total

# Fast version: calculates the sum only once
def fast_sum():
    total = 0
    list_sum = sum(my_list)

    for i in range(len(my_list)):
        total += list_sum

    return total

# Check that both results match
print(slow_sum() == fast_sum())

# Compare execution times
%timeit slow_sum()
%timeit fast_sum()

True
551 ms ± 3.76 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
386 μs ± 40.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


The slow version calculates the sum of the entire list during
every iteration of the loop.

The fast version calculates the sum only once and stores it
in a variable.

This makes the fast version much more efficient because it
avoids repeating the same calculation thousands of times.

✅ **Your Turn**: Try squaring numbers with a Python loop, a list comprehension, and NumPy at several sizes (say 1,000, 100,000, and 1,000,000). How does the **gap** between them change as the size grows?

In [22]:
# Your turn here
# Size 1,000
numbers = list(range(1_000))
numbers_array = np.arange(1_000)

%timeit [value ** 2 for value in numbers]
%timeit numbers_array ** 2

# Size 100,000
numbers = list(range(100_000))
numbers_array = np.arange(100_000)

%timeit [value ** 2 for value in numbers]
%timeit numbers_array ** 2

# Size 1,000,000
numbers = list(range(1_000_000))
numbers_array = np.arange(1_000_000)

%timeit [value ** 2 for value in numbers]
%timeit numbers_array ** 2

37.5 μs ± 423 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
1.07 μs ± 72.3 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
4.18 ms ± 59.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
30.4 μs ± 441 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
60.9 ms ± 1.13 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
2.03 ms ± 215 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


NumPy is generally faster than list comprehensions when working
with large amounts of data because it performs calculations
on entire arrays at once.

As the number of values increases, the performance difference
usually becomes more noticeable.

## 8. Finding the Slow Part: Profiling
`%time` and `%timeit` tell you how long a whole block takes. A **profiler** tells you **where** inside it the time goes. `%prun` runs a statement and reports the time spent in each function it called.

The pipeline below calls two helper functions for every value. Which one is the bottleneck?

In [23]:
def double(value):
    return value * 2

def is_banned(value, banned):
    return value in banned          # searching a list checks items one by one

def pipeline(values, banned):
    kept = []
    for v in values:
        if not is_banned(v, banned):
            kept.append(double(v))
    return kept

banned_list = list(range(0, 20_000, 7))
%prun -s cumulative -l 6 pipeline(range(20_000), banned_list)

         54288 function calls in 0.334 seconds

   Ordered by: cumulative time
   List reduced from 7 to 6 due to restriction <6>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.334    0.334 {built-in method builtins.exec}
        1    0.000    0.000    0.334    0.334 <string>:1(<module>)
        1    0.008    0.008    0.334    0.334 3096098862.py:7(pipeline)
    20000    0.323    0.000    0.323    0.000 3096098862.py:4(is_banned)
    17142    0.002    0.000    0.002    0.000 {method 'append' of 'list' objects}
    17142    0.001    0.000    0.001    0.000 3096098862.py:1(double)

How to read it:
- **ncalls**: how many times the function was called
- **tottime**: time spent **inside** that function itself
- **cumtime**: time inside it **plus** everything it called

`is_banned` dominates. Searching a **list** with `in` checks items one at a time, but a **set** finds an item almost instantly. That's a one-word fix:

In [24]:
banned_set = set(banned_list)

print(pipeline(range(20_000), banned_list) == pipeline(range(20_000), banned_set))
%timeit pipeline(range(20_000), banned_list)
%timeit pipeline(range(20_000), banned_set)

True
341 ms ± 6.47 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
1.74 ms ± 4.22 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


(`-s cumulative` sorts by cumulative time and `-l 6` shows only the top 6 rows. In VS Code, the report appears in the cell's output.)

Profile only code you've **already measured** to be slow. Profiling every two-line example is like using a metal detector to look for a paperclip.

### Line by Line (Optional)
When `%prun` points at one function but not at the slow **line**, the `line_profiler` package can time each line separately. It isn't part of the course requirements, so install it first with `%pip` (notebook 01):

In [25]:
%pip install line_profiler

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
%load_ext line_profiler
%lprun -f pipeline pipeline(range(20_000), banned_list)

The `% Time` column shows which line is the bottleneck. If you can't install `line_profiler`, split the function into pieces and `%timeit` each one.

✅ **Your Turn**: Write a function that runs **two** loops, one cheap (for example, summing a list) and one expensive (for example, checking whether each of 5,000 numbers is `in` a 5,000-item list). Use `%prun` to show which part takes the most time.

In [26]:
# Your turn here
def test_performance():
    numbers = list(range(5_000))
    banned = list(range(5_000))

    # First loop: summing numbers
    total = 0

    for number in numbers:
        total += number

    # Second loop: checking if numbers exist in the list
    found = []

    for number in numbers:
        if number in banned:
            found.append(number)

    return total, found

# Profile the function
%prun -s cumulative -l 10 test_performance()

         5064 function calls in 0.083 seconds

   Ordered by: cumulative time
   List reduced from 38 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.067    0.067 {built-in method builtins.exec}
        1    0.000    0.000    0.067    0.067 <string>:1(<module>)
        1    0.066    0.066    0.067    0.067 3669230306.py:2(test_performance)
        2    0.015    0.008    0.015    0.008 {method '__exit__' of 'sqlite3.Connection' objects}
     5000    0.001    0.000    0.001    0.000 {method 'append' of 'list' objects}
        2    0.000    0.000    0.000    0.000 traitlets.py:726(__set__)
        2    0.000    0.000    0.000    0.000 traitlets.py:3627(set)
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}
        2    0.000    0.000    0.000    0.000 traitlets.py:707(set)
        2    0.000    0.000    0.000    0.000 traitlets.py:736(_validate)

The second loop is the slowest part because it searches through
a list every time it checks whether a number exists.

The first loop is faster because it only adds numbers together.

Searching a list requires checking its elements one by one.
Using a set instead would make searching much faster.

## 9. Measuring Memory
Fast code can still be a problem if it uses too much memory. NumPy arrays tell you exactly how much space their data takes:

In [27]:
values = np.arange(1_000_000)

print('shape:', values.shape)
print('dtype:', values.dtype)
print('bytes:', f'{values.nbytes:,}')    # 1,000,000 values x 8 bytes each

shape: (1000000,)
dtype: int64
bytes: 8,000,000


The **dtype** decides the size of each value. If your numbers fit in a smaller type, converting with `.astype()` cuts memory, but check the range first! `int8` can only hold -128 to 127.

In [28]:
for dtype in ['int64', 'int32', 'int16', 'float32']:
    print(f'{dtype:8} {values.astype(dtype).nbytes:>10,} bytes')

int64     8,000,000 bytes
int32     4,000,000 bytes
int16     2,000,000 bytes
float32   4,000,000 bytes


### Lists Are Bigger Than They Look
`sys.getsizeof()` reports an object's size, but for a list it counts only the list's internal "pointers", **not** the numbers themselves. Treat it as a starting point, not a full memory audit:

In [29]:
as_list = list(range(1_000_000))

print(f'list (pointers only):    {sys.getsizeof(as_list):>12,} bytes')
print(f'list (with the ints):    {sys.getsizeof(as_list) + sum(sys.getsizeof(n) for n in as_list):>12,} bytes')
print(f'NumPy array (int64):     {values.nbytes:>12,} bytes')

list (pointers only):       8,000,056 bytes
list (with the ints):      36,000,056 bytes
NumPy array (int64):        8,000,000 bytes


✅ **Your Turn**: Create a NumPy array of one million random **floats** with `np.random.random(1_000_000)`. Check its `shape`, `dtype`, and `nbytes`. Then convert it to `float32` and compare the memory. Print the first 3 values of each. What did you give up for the smaller size?

In [30]:
# Your turn here
# Create one million random floats
random_values = np.random.random(1_000_000)

# Display original array information
print("Original array:")
print("Shape:", random_values.shape)
print("Data type:", random_values.dtype)
print("Memory:", f"{random_values.nbytes:,}", "bytes")

# Convert to float32
smaller_values = random_values.astype('float32')

print("\nFloat32 array:")
print("Shape:", smaller_values.shape)
print("Data type:", smaller_values.dtype)
print("Memory:", f"{smaller_values.nbytes:,}", "bytes")

# Print the first 3 values
print("\nFirst 3 original values:")
print(random_values[:3])

print("\nFirst 3 float32 values:")
print(smaller_values[:3])

Original array:
Shape: (1000000,)
Data type: float64
Memory: 8,000,000 bytes

Float32 array:
Shape: (1000000,)
Data type: float32
Memory: 4,000,000 bytes

First 3 original values:
[0.82333891 0.40795565 0.57547238]

First 3 float32 values:
[0.8233389  0.40795565 0.57547235]


The float32 array uses less memory than the original float64 array.
It uses approximately half the memory because each value takes
4 bytes instead of 8 bytes.

The trade-off is precision. Float32 stores fewer decimal digits,
so some values may be slightly different from the original array.

## 10. Working with Data Efficiently
The same ideas apply to Pandas (notebook 06). First, let's make a realistic-sized CSV file of 100,000 sales records:

In [31]:
np.random.seed(3603)
n = 100_000
sales = pd.DataFrame({
    'date': pd.date_range('2026-01-01', periods=n, freq='min').strftime('%Y-%m-%d'),
    'department': np.random.choice(['Books', 'Electronics', 'Home', 'Sports'], size=n),
    'store_id': np.random.randint(1, 50, size=n),
    'units': np.random.randint(1, 10, size=n),
    'amount': np.random.uniform(5, 500, size=n).round(2),
    'notes': np.random.choice(['', 'gift', 'returned', 'online order'], size=n),
})
sales.to_csv(OUT / 'sales.csv', index=False)
sales.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   date        100000 non-null  str    
 1   department  100000 non-null  str    
 2   store_id    100000 non-null  int32  
 3   units       100000 non-null  int32  
 4   amount      100000 non-null  float64
 5   notes       100000 non-null  str    
dtypes: float64(1), int32(2), str(3)
memory usage: 17.7 MB


💡 `memory_usage='deep'` counts the real size of text columns. Without it, Pandas only estimates them.

### Read Only the Columns You Need
`usecols` (notebook 06) skips the other columns entirely, which saves both time and memory:

In [32]:
%time full = pd.read_csv(OUT / 'sales.csv')
%time slim = pd.read_csv(OUT / 'sales.csv', usecols=['department', 'amount'])

print(f'full: {full.memory_usage(deep=True).sum():>12,} bytes')
print(f'slim: {slim.memory_usage(deep=True).sum():>12,} bytes')

CPU times: total: 62.5 ms
Wall time: 69.4 ms
CPU times: total: 46.9 ms
Wall time: 41.1 ms
full:   18,925,944 bytes
slim:    6,349,762 bytes


### Use `category` for Repeated Text
A text column with only a few distinct values, like `department`, can be stored as a **category**. Each distinct value is stored once, and every row just holds a small code:

In [33]:
as_text = pd.read_csv(OUT / 'sales.csv', usecols=['department'])
as_category = pd.read_csv(OUT / 'sales.csv', usecols=['department'], dtype={'department': 'category'})

print(f'text:     {as_text.memory_usage(deep=True).sum():>10,} bytes')
print(f'category: {as_category.memory_usage(deep=True).sum():>10,} bytes')

text:      5,549,762 bytes
category:    100,526 bytes


### Process Huge Files in Chunks
If a file is too big to fit in memory, `chunksize` reads it a piece at a time, so you can keep a running total:

In [34]:
total_revenue = 0
for chunk in pd.read_csv(OUT / 'sales.csv', chunksize=25_000):
    total_revenue += chunk['amount'].sum()

print(f'Total revenue: ${total_revenue:,.2f}')
print('Check:         $' + f"{full['amount'].sum():,.2f}")

Total revenue: $25,281,028.41
Check:         $25,281,028.41


### Avoid Row-by-Row Loops on DataFrames
`.iterrows()` loops over a DataFrame one row at a time, which is the Pandas version of the slow Python loop. Whole-column operations are vectorized, just like NumPy:

In [35]:
%%time
discounted = []
for _, row in full.iterrows():
    discounted.append(row['amount'] * 0.90)

CPU times: total: 2.56 s
Wall time: 2.56 s


In [ ]:
%%time
full['discounted'] = full['amount'] * 0.90

✅ **Your Turn**:
1. Read `sales.csv` with `usecols` for just `store_id` and `units`. Compare its time and memory to `full`.
2. Use `.iterrows()` to compute `units * amount` for every row (on `full.head(10_000)`, to keep it quick), then do the same thing with one column operation. Time both and check that the results match.

In [38]:
# Your turn here
# Read only the required columns
%time slim_store = pd.read_csv(
    OUT / 'sales.csv',
    usecols=['store_id', 'units']
)

# Display memory usage
print("Full DataFrame memory:")
print(f"{full.memory_usage(deep=True).sum():,} bytes")

print("\nSmaller DataFrame memory:")
print(f"{slim_store.memory_usage(deep=True).sum():,} bytes")

IndentationError: unexpected indent (2935884647.py, line 4)

In [39]:
# Use the first 10,000 rows
small_sales = full.head(10_000)

# Slow version: using iterrows()
%%time
loop_result = []

for index, row in small_sales.iterrows():
    loop_result.append(row['units'] * row['amount'])

# Fast version: using a column operation
%%time
vector_result = small_sales['units'] * small_sales['amount']

# Check if the results match
print(np.allclose(loop_result, vector_result))

UsageError: Line magic function `%%time` not found.


The vectorized operation is faster because Pandas performs
the calculation on the entire column instead of processing
one row at a time.

Both approaches produce the same results, but the vectorized
version is more efficient.

## 11. Trustworthy Timings
Notebook timings can mislead when cells depend on leftover variables, or when something was **cached** by an earlier run. For results you'll report:
1. **Restart** the kernel (notebook 02).
2. Run the **setup** cells.
3. Run the **timing** cell, then repeat the measurement.
4. Record the **typical** result, not the luckiest one.

Also, keep setup **out** of the timed line. `%timeit np.sqrt(values)` measures the square root, not the creation of `values`.

✅ **Your Turn**: Restart the kernel, re-run the Setup cell and the squaring comparison in section 6, and compare the new speedup to your first run. In the cell below, explain why restarting makes results more trustworthy, and why the two numbers aren't identical.

*Your explanation here.*

Restarting the kernel makes the measurements more trustworthy
because it clears previous variables and temporary results.

The two timing results are not exactly identical because
execution time can change depending on background processes,
computer activity, and other factors.

NumPy is generally faster because it performs calculations
using optimized operations instead of Python loops.

---
### Summary
- **Measure, don't guess.** Find the bottleneck, improve it, then measure again.
- `%time`/`%%time` give a quick single measurement (wall time is what you waited). `%timeit`/`%%timeit` run many times for reliable comparisons. Use `-o` to save the result and compute a speedup.
- Variables created inside `%%timeit` don't survive. `%%time` keeps them.
- Compare fairly: same input, same work, same result, setup excluded.
- NumPy and Pandas **vectorized** operations beat Python loops on large data, and are often clearer too. Never loop over an array or use `.iterrows()` when a whole-column operation exists.
- Move repeated work out of loops. Watch for runtime that grows faster than the data.
- `%prun` finds the slow function, and `%lprun` finds the slow line. Lists search slowly and sets search fast.
- Check memory with `.nbytes`, `.astype()`, and `memory_usage(deep=True)`. Load less data with `usecols`, `category`, and `chunksize`.

## 🏋️ Practice

### Practice 1 — Time a Sort (easy)
Create a list of 100,000 random numbers (hint: `random.sample(range(1_000_000), 100_000)` after `import random`) and use `%timeit` to measure how long `sorted(...)` takes on it.

In [41]:
import random
# Your code here
# Generate 100,000 random numbers
random_numbers = random.sample(range(1_000_000), 100_000)

# Measure sorting time
%timeit sorted(random_numbers)

25.6 ms ± 2.33 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


### Practice 2 — Loop vs. Comprehension (medium)
Using the `numbers` list below, compare `%timeit` for a `for` loop that builds a squared list with `.append()` vs. a list comprehension that does the same thing. Check that both give the same list, and use `-o` to print how many times faster the winner is.

In [42]:
numbers = list(range(1_000))
# Time the for-loop version, then the comprehension version

# Function using a for loop
def square_loop(numbers):
    result = []

    for number in numbers:
        result.append(number ** 2)

    return result

# Function using a list comprehension
def square_comprehension(numbers):
    return [number ** 2 for number in numbers]

# Check that both results match
print(square_loop(numbers) == square_comprehension(numbers))

# Measure execution times
t_loop = %timeit -o square_loop(numbers)
t_comprehension = %timeit -o square_comprehension(numbers)

# Compare performance
print(f"List comprehension is {t_loop.average / t_comprehension.average:.2f}x faster.")

True
46.8 μs ± 767 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
47 μs ± 6.8 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
List comprehension is 1.00x faster.


### Practice 3 — Three-Way Comparison (harder)
Compare `%timeit` for squaring 100,000 numbers three ways: a `for` loop over `list(range(100_000))`, a list comprehension over the same list, and NumPy on `np.arange(100_000)`. Which is fastest? Which is slowest? Then repeat the loop version over the **NumPy array** instead of the list, and explain the result (see section 6).

In [43]:
# Your three timing comparisons here
# Create the data
numbers_list = list(range(100_000))
numbers_array = np.arange(100_000)

# Python loop
def square_loop(values):
    result = []

    for value in values:
        result.append(value ** 2)

    return result

# List comprehension
def square_comprehension(values):
    return [value ** 2 for value in values]

# NumPy
def square_numpy(values):
    return values ** 2

# Check that all three approaches give the same result
loop_result = square_loop(numbers_list)
comprehension_result = square_comprehension(numbers_list)
numpy_result = square_numpy(numbers_array)

print(loop_result == comprehension_result)
print(np.array_equal(np.array(loop_result), numpy_result))

# Measure execution times
t_loop = %timeit -o square_loop(numbers_list)

t_comprehension = %timeit -o square_comprehension(numbers_list)

t_numpy = %timeit -o square_numpy(numbers_array)

# Display the results
print("\nPerformance comparison:")

print(f"Python loop: {t_loop.average:.6f} seconds")
print(f"List comprehension: {t_comprehension.average:.6f} seconds")
print(f"NumPy: {t_numpy.average:.6f} seconds")

# Find the fastest and slowest approaches
times = {
    'Python loop': t_loop.average,
    'List comprehension': t_comprehension.average,
    'NumPy': t_numpy.average
}

print("\nFastest:", min(times, key=times.get))
print("Slowest:", max(times, key=times.get))

True
True
6.45 ms ± 295 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
6.72 ms ± 584 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
32.8 μs ± 407 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)

Performance comparison:
Python loop: 0.006448 seconds
List comprehension: 0.006722 seconds
NumPy: 0.000033 seconds

Fastest: NumPy
Slowest: List comprehension


In [44]:
# Python loop over a NumPy array
t_array_loop = %timeit -o square_loop(numbers_array)

print(f"Python loop over NumPy array: {t_array_loop.average:.6f} seconds")
print(f"NumPy vectorized: {t_numpy.average:.6f} seconds")

9.42 ms ± 742 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Python loop over NumPy array: 0.009422 seconds
NumPy vectorized: 0.000033 seconds


NumPy is generally the fastest because it performs calculations
on the entire array using optimized compiled code.

The Python loop is generally the slowest because it processes
each value individually.

A loop over a NumPy array can be even slower than a loop over
a regular Python list because each array value must be accessed
and converted into a Python object.

List comprehensions are usually faster than regular for loops
because they have less Python overhead.

## 🔥 Challenge (Optional) — Where's the Line?
For very small inputs, a plain Python loop can actually be just as fast as (or faster than) NumPy, because NumPy has a small fixed overhead per call. Find roughly how small an array has to be before the loop wins, by timing both approaches at a few different sizes (e.g. 10, 100, 1,000, 100,000).

In [ ]:
# Your investigation here


## 📦 Check Your Output Files
This notebook's only output file is `sales.csv`, the generated practice data. Upload `08-OutputFiles` along with this notebook.

In [45]:
for path in sorted(OUT.rglob('*')):
    print(path, f'({path.stat().st_size:,} bytes)')

08-OutputFiles\sales.csv (3,800,802 bytes)
